# 06 — Limpieza profesional con criterio — Airbnb Listings

Orden no intercambiable: **tipos → duplicados → nulos → formato**.
Sin tipos correctos, `isnull()` puede fallar en silencio y las estadísticas de relleno estarán sesgadas.


## Setup

In [ ]:
import pandas as pd
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()
# latin-1 porque el archivo tiene bytes no-UTF8 (caracteres de varias ciudades del mundo)
df = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                 encoding='latin-1', low_memory=False)

print(f'Shape: {df.shape}')
print(df.dtypes)


Shape: (279712, 33)
listing_id                       int64
name                               str
host_id                          int64
host_since                         str
host_location                      str
host_response_time                 str
host_response_rate             float64
host_acceptance_rate           float64
host_is_superhost                  str
host_total_listings_count      float64
host_has_profile_pic               str
host_identity_verified             str
neighbourhood                      str
district                           str
city                               str
latitude                       float64
longitude                      float64
property_type                      str
room_type                          str
accommodates                     int64
bedrooms                       float64
amenities                          str
price                            int64
minimum_nights                   int64
maximum_nights                   int64
revie

---
## 1 — Tipos correctos: Corregir objetos a sus debidos dtypes


Primer paso siempre. Cuando una collumn es un object o (string) en vez de ser el **tipo correcto de dato** la operacion, sobre consola **no falla**, devuelven un resultafo incorrecto sin avisar.

Es decir: no existe error, el resultado está mal, pero el programador no lo sabrá. 

Por eso la importancia de cambiar los dtypes. Hacer lo correcto para futuras operaciones.

1 - Cambiar fecha a formato ISO 8601 
2 - Cambiar Las filas con caracteres normales que refieren a bolleanos a true y false

In [ ]:
#Como regla crear siempre el copy para operar sobre y no depender de 
# un dataframe solamente, los datos pueden necesitar mostrarse desde cero en caso de equivoco
df = df.copy()

# Poder encontrar siempre la fila que representa una 
# fecha y cambiar el dtype o tipo de dato de esa columna para poder ralizar operaciones
# en este caso host_since

# host_since llega como string '2011-12-03' — con pandas se convierte en datetame para, por ejemplo, 
# ejecutar un analisis sobre antiguedad
df['host_since'] = pd.to_datetime(df['host_since'])

# Columnas con valores 't'/'f' — Airbnb exporta booleans como strings
bool_cols = ['host_is_superhost', 'host_has_profile_pic',
             'host_identity_verified', 'instant_bookable']

for col in bool_cols:
     # map() deja NaN donde no hay 't' ni 'f' — correcto, hay nulos en esas columnas
     # La idea es pasar los objetos a booleanos, tenerlos listo para operaciones
    df[col] = df[col].map({'t': True, 'f': False})

# Siempre presentar la información de manera bastante clara para no leer código, solo salidas, o consola
print('Tipos corregidos:')
print(df[['host_since'] + bool_cols].dtypes)
print()
print('Distribución superhost:')
print(df['host_is_superhost'].value_counts(dropna=False))


Tipos corregidos:
host_since                datetime64[us]
host_is_superhost                 object
host_has_profile_pic              object
host_identity_verified            object
instant_bookable                  object
dtype: object

Distribución superhost:
host_is_superhost
NaN    279712
Name: count, dtype: int64


---
## 2 — Duplicados

Antes de tratar nulos: si una fila duplicada tiene nulos y la otra no, eliminar después del relleno conservaría datos fabricados.

Imagina que una fila duplicada tiene valores iguales, a excepción de un valor, que es nulo.

Si se trata antes este valor duplicado ser "tratado" o igualarlo a su "id" identico significa que se puede identificar esa fila duplicada y así borrarla. Caso contrario esa fila seguiría molestando resultados.


In [ ]:
# Identificación primaria sobre los duplicados, lo mas impoerante es hacerlo sobre todo el dset
# "Documentar" o explicar bien a través de prints la cantidad, donde están...
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')

# Verificar también duplicados por listing_id — puede haber filas distintas
# con el mismo id si el export tiene versiones del mismo alojamiento
n_dup_id = df.duplicated(subset='listing_id').sum()
print(f'listing_id duplicados: {n_dup_id}')


if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Shape tras drop_duplicates: {df.shape}')
else:
    print('Sin duplicados — no se modifica el DataFrame')


---
## 3 — Nulos: diagnóstico

Ver el mapa completo antes de tomar ninguna decisión.

In [ ]:
nulos = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('Porcentaje de nulos por columna:')
print(nulos[nulos > 0].to_string())


### 3.1 — Columna con >50% nulos → eliminar

`district` tiene 86.8% de nulos. Rellenar esa columna sería más invención que dato. Se elimina.

In [ ]:
UMBRAL_NULOS = 0.50
cols_a_eliminar = [c for c in df.columns if df[c].isnull().mean() > UMBRAL_NULOS]
print(f'Columnas con >{UMBRAL_NULOS*100:.0f}% nulos: {cols_a_eliminar}')

df = df.drop(columns=cols_a_eliminar)
print(f'Shape tras eliminar columnas vacías: {df.shape}')


### 3.2 — `bedrooms` (10.5% nulos) → fillna con mediana por tipo de propiedad

La mediana de bedrooms de un `Entire apartment` no es la misma que la de un `Private room in house`.
Rellenar con la mediana global mezclaría esas distribuciones.

In [ ]:
mediana_por_tipo = df.groupby('property_type')['bedrooms'].transform('median')
# transform devuelve una Series con el mismo índice que df — alineación automática

nulos_antes = df['bedrooms'].isnull().sum()
df['bedrooms'] = df['bedrooms'].fillna(mediana_por_tipo)

# Los property_type muy raros pueden no tener mediana — rellenar el residuo con la global
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())

nulos_despues = df['bedrooms'].isnull().sum()
print(f'Nulos bedrooms: {nulos_antes} → {nulos_despues}')


### 3.3 — `host_response_rate` y `host_acceptance_rate` (40-46% nulos)

Los nulos en estas columnas corresponden en su mayoría a hosts nuevos o con pocas reservas.
Rellenar con 0 sería engañoso (no es que no respondan, es que no hay historial).
Se rellena con la mediana del grupo de ciudad — los mercados tienen dinámicas distintas.

In [ ]:
for col in ['host_response_rate', 'host_acceptance_rate']:
    mediana_ciudad = df.groupby('city')[col].transform('median')
    nulos_antes = df[col].isnull().sum()
    df[col] = df[col].fillna(mediana_ciudad)
    # Residuo — ciudades con muy pocos registros sin valor
    df[col] = df[col].fillna(df[col].median())
    print(f'{col}: {nulos_antes:,} nulos → {df[col].isnull().sum():,}')


### 3.4 — `review_scores_*` (32.8% nulos) → mantener como NaN

Los nulos en review scores no son errores: los listings nuevos no tienen reviews todavía.
Imputar scores inventados distorsionaría cualquier análisis de calidad.
La ausencia de dato es el dato correcto aquí.

In [ ]:
review_cols = [c for c in df.columns if c.startswith('review_scores')]
print('Columnas de reviews — nulos justificados (listings sin historial):')
print(df[review_cols].isnull().sum().to_string())
print()
# Verificar: ¿los nulos en reviews coinciden con listings sin host_since reciente?
sin_review = df[df['review_scores_rating'].isnull()]
con_review  = df[df['review_scores_rating'].notna()]
print(f'Media noches mínimas — sin review: {sin_review["minimum_nights"].median():.0f}')
print(f'Media noches mínimas — con review:  {con_review["minimum_nights"].median():.0f}')


---
## 4 — Inconsistencias de formato

### 4.1 — `host_response_time` → categorías inconsistentes

In [ ]:
print('Valores únicos host_response_time:')
print(df['host_response_time'].value_counts(dropna=False))

# Normalizar a minúsculas y sin espacios extra para comparaciones seguras
df['host_response_time'] = (
    df['host_response_time']
    .str.strip()
    .str.lower()
)
print()
print('Tras normalización:')
print(df['host_response_time'].value_counts(dropna=False))


### 4.2 — `amenities` → string JSON → métrica numérica

La columna llega como un string que parece una lista. Se puede transformar en un conteo de amenities por listing.

In [ ]:
# Muestra del formato original
print(df['amenities'].iloc[0])
print()

# Contar elementos — split por coma aproxima el número de amenities
# (no parsear como JSON porque el formato puede variar entre filas)
df['amenities_count'] = (
    df['amenities']
    .fillna('')
    .str.split(',')
    .apply(len)
)

print('Distribución de amenities_count:')
print(df['amenities_count'].describe().round(1))


---
## 5 — Función de limpieza reutilizable

In [ ]:
def limpiar_airbnb(df: 'pd.DataFrame', umbral_nulos: float = 0.50) -> 'pd.DataFrame':
    df = df.copy()

    # Tipos
    df['host_since'] = pd.to_datetime(df['host_since'])
    for col in ['host_is_superhost', 'host_has_profile_pic',
                'host_identity_verified', 'instant_bookable']:
        if col in df.columns:
            df[col] = df[col].map({'t': True, 'f': False})

    # Columnas con demasiados nulos
    cols_vacias = [c for c in df.columns if df[c].isnull().mean() > umbral_nulos]
    df = df.drop(columns=cols_vacias)

    # Duplicados
    df = df.drop_duplicates()

    # Nulos
    if 'bedrooms' in df.columns:
        df['bedrooms'] = df['bedrooms'].fillna(
            df.groupby('property_type')['bedrooms'].transform('median')
        ).fillna(df['bedrooms'].median())

    for col in ['host_response_rate', 'host_acceptance_rate']:
        if col in df.columns:
            df[col] = df[col].fillna(
                df.groupby('city')[col].transform('median')
            ).fillna(df[col].median())

    # Formato
    if 'host_response_time' in df.columns:
        df['host_response_time'] = df['host_response_time'].str.strip().str.lower()

    if 'amenities' in df.columns:
        df['amenities_count'] = df['amenities'].fillna('').str.split(',').apply(len)

    print(f'Dataset limpio: {df.shape[0]:,} filas, {df.shape[1]} columnas')
    print(f'Eliminadas: {cols_vacias}')
    return df


# Cargar de nuevo y aplicar la función completa
df_raw = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                     encoding='latin-1', low_memory=False)
df_clean = limpiar_airbnb(df_raw)
print()
print('Nulos restantes:')
print((df_clean.isnull().mean() * 100).round(1).sort_values(ascending=False).head(10))


---
## Resumen de decisiones

| Columna | Nulos | Decisión | Justificación |
|---------|-------|----------|--------------|
| `district` | 86.8% | Eliminar columna | >50% — rellenar sería invención |
| `bedrooms` | 10.5% | Mediana por `property_type` | La mediana global mezcla tipos de propiedad distintos |
| `host_response_rate` | 46% | Mediana por ciudad | Los nulos son hosts sin historial, no errores |
| `review_scores_*` | 32.8% | Mantener NaN | Ausencia justificada: listings sin reviews aún |
| `host_is_superhost` | t/f → bool | Corrección de tipo | Los strings no operan como booleanos |
| `host_since` | str → datetime | Corrección de tipo | Sin datetime no se puede calcular antigüedad |
